# Vault Auto-Unseal with AWS KMS — On-Prem Vault accessing AWS KMS

This notebook configures Vault's `seal "awskms"` when **Vault runs on-premises** (Minikube outside AWS) and auto-unseals with an **AWS KMS** key. It was verified against **Vault Enterprise 2.1.0** (`awskms/v2 v2.0.11`, `awsutil v0.3.0`). CloudTrail in `eu-west-3` showed `kms:Encrypt` / `kms:Decrypt` / `kms:DescribeKey` as `AssumedRole` `vault-kms-unseal`, not as IAM user `vault-autounseal`.

### Least privilege

The bootstrap IAM user can only call `sts:AssumeRole` on `vault-kms-unseal`. That role, and the KMS key policy, grant `kms:Encrypt`, `kms:Decrypt`, and `kms:DescribeKey`. The role trust policy names the user ARN as `Principal` (same-account). Kubernetes mounts a shared credentials file; Vault Pods do not receive `AWS_ACCESS_KEY_ID` / `AWS_SECRET_ACCESS_KEY`.

### Configuration that worked

Shared credentials file (`/vault/userconfig/aws/credentials`):

```ini
[default]
aws_access_key_id=...
aws_secret_access_key=...

[vault-bootstrap]
role_arn=arn:aws:iam::<account>:role/vault-kms-unseal
role_session_name=vault-auto-unseal
source_profile=default
```

Seal stanza plus Pod env:

```hcl
seal "awskms" {
  region                = "eu-west-3"
  kms_key_id            = "<kms-key-id>"
  shared_creds_filename = "/vault/userconfig/aws/credentials"
  shared_creds_profile  = "vault-bootstrap"
  role_arn              = "arn:aws:iam::<account>:role/vault-kms-unseal"
  role_session_name     = "vault-auto-unseal"
}
```

```text
AWS_SHARED_CREDENTIALS_FILE=/vault/userconfig/aws/credentials
```

Do not set `AWS_PROFILE`. `role_arn` / `source_profile` in the INI file are **ignored** by `awsutil` v0.3.0's shared-credentials provider (static keys only). `[vault-bootstrap]` exists so that provider finds no `aws_access_key_id` and fails. `role_arn` in the stanza then adds AssumeRole. The inner STS session reads `[default]` through `AWS_SHARED_CREDENTIALS_FILE`.

### Configurations that failed

1. Static keys in the profile the wrapper selects **and** `role_arn` in the stanza — the chain picks the first valid provider, so KMS runs as the IAM user and never assumes the role.
2. `role_arn` / `source_profile` only in the INI file, omitted from the stanza — shared-credentials finds no keys and AssumeRole is never added → `NoCredentialProviders`.
3. Role trust of account-root plus `aws:PrincipalArn` — same-account AssumeRole then depends on the user's identity policy, which a permissions boundary can deny permanently. Trust the user ARN in `Principal` instead.

### STS token lifetime (tested)

`awsutil` v0.3.0 snapshots the STS credentials into a `StaticProvider` at process start. A running Pod does **not** call `AssumeRole` again. The role session lasts **1 hour**. A new `ASIA…` access key appears in CloudTrail only after the process restarts (`kubectl delete pod`). Waiting out the hour without a restart keeps the same key until it expires; KMS then fails until the next start. For automatically renewable credentials, use the Web Identity/OIDC notebook.

### Architecture

```
┌─ On-Prem (K8s Cluster) ─────────┐      ┌─ AWS Account ────────────────────────┐
│                                  │      │                                      │
│  Vault Pod                       │      │  IAM user: vault-autounseal          │
│   ├─ [default] static keys       │ STS  │   └─ sts:AssumeRole only             │
│   ├─ [vault-bootstrap] no keys   │─────▶│  Role: vault-kms-unseal             │
│   └─ seal "awskms"               │      │   └─ KMS Encrypt/Decrypt/Describe   │
│        profile vault-bootstrap   │      │                                      │
│        role_arn ─────────────────┼─────▶│  KMS Key: vault-auto-unseal          │
│        kms_key_id ───────────────┼─────▶│   └─ Key policy grants the role      │
│                                  │      │                                      │
└──────────────────────────────────┘      └──────────────────────────────────────┘
```

`role_external_id` is not used: the AWS KMS wrapper option parser does not expose it. Environment merging can supply `AWSKMS_WRAPPER_KEY_ID`, `AWS_ACCESS_KEY_ID`, `AWS_DEFAULT_REGION`, `AWS_KMS_ENDPOINT`, `AWS_REGION`, `AWS_SECRET_ACCESS_KEY`, `AWS_SESSION_TOKEN`, and `VAULT_AWSKMS_SEAL_KEY_ID`; this notebook does not put access keys in the Vault process environment.

Source references: [Vault Enterprise environment merge](https://github.com/hashicorp/vault-enterprise/blob/03515e06228c7eb8877e3043e87cd2ccf58d963a/internalshared/configutil/env_var_util.go#L25), [Enterprise AWS KMS wrapper construction](https://github.com/hashicorp/vault-enterprise/blob/03515e06228c7eb8877e3043e87cd2ccf58d963a/internalshared/configutil/kms_awskms_ent.go#L19), [AWS wrapper option parsing](https://github.com/hashicorp/go-kms-wrapping/blob/d4ca45ec7310b5efea9a72993046cf896ff69550/wrappers/awskms/options.go#L50), and [awsutil v0.3.0 credential chain](https://github.com/hashicorp/go-secure-stdlib/blob/awsutil/v0.3.0/awsutil/generate_credentials.go).

## 1. Load base AWS credentials

In [ ]:
# Read aws credentials from csv file and set as environment variables
import csv, os
with open('vault_test_accessKeys.csv', 'r', encoding='utf-8-sig') as csvfile:
    reader = csv.DictReader(csvfile)
    creds = next(reader)
    access_key = creds['Access key ID'].strip()
    secret_key = creds['Secret access key'].strip()
    os.environ['AWS_ACCESS_KEY_ID'] = access_key
    os.environ['AWS_SECRET_ACCESS_KEY'] = secret_key
    # The CSV contains a long-lived IAM key; never combine it with a stale STS token.
    os.environ.pop('AWS_SESSION_TOKEN', None)
    os.environ.pop('AWS_SECURITY_TOKEN', None)
    print(f"Access Key: {access_key[:8]}...")
    print(f"Credentials loaded.")

## 2. Create KMS Key, bootstrap IAM user, and AssumeRole role

Vault runs **on-prem** but needs access to an AWS KMS key for auto-unseal.  
We create:
- A bootstrap **IAM user** with programmatic access and only `sts:AssumeRole` permission
- An **IAM role** trusted by that user and authorized to use the KMS key
- A **KMS key policy** that grants cryptographic operations to the role

Vault selects profile `vault-bootstrap` (no static keys) and sets `role_arn` in the seal stanza, so the shared-credentials provider fails closed and AssumeRole runs with the `[default]` keys.

In [ ]:
import boto3
import json
import os
import time
from botocore.exceptions import ClientError

REGION = 'eu-west-3'
IAM_USER_NAME = 'vault-autounseal'
KMS_ALIAS = 'alias/vault-auto-unseal'
POLICY_NAME = 'vault-autounseal-kms-policy'
ROLE_NAME = 'vault-kms-unseal'
USER_ASSUME_POLICY_NAME = 'vault-assume-kms-role'

try:
    kms_client = boto3.client('kms', region_name=REGION)
    iam_client = boto3.client('iam')
    sts_client = boto3.client('sts')
    caller_identity = sts_client.get_caller_identity()
    account_id = caller_identity['Account']
    caller_arn = caller_identity['Arn']
    target_user_arn = f"arn:aws:iam::{account_id}:user/{IAM_USER_NAME}"
    if caller_arn == target_user_arn:
        raise RuntimeError(
            f'The provisioning credentials belong to {target_user_arn}. '
            'Use a separate administrator/provisioner identity; rotating the '
            'bootstrap user while authenticating as that user invalidates the run.'
        )

    # ─── 1. Create or reuse IAM User ───────────────────────────────────────
    try:
        iam_client.create_user(
            UserName=IAM_USER_NAME,
            Tags=[{'Key': 'Purpose', 'Value': 'vault-auto-unseal'}]
        )
        print(f"✓ IAM user '{IAM_USER_NAME}' created")
        time.sleep(30)
    except ClientError as e:
        if e.response['Error']['Code'] == 'EntityAlreadyExists':
            print(f"✓ IAM user '{IAM_USER_NAME}' already exists — reusing")
            # Delete old bootstrap keys. The guard above guarantees none is
            # the credential currently provisioning this environment.
            for key in iam_client.list_access_keys(UserName=IAM_USER_NAME)['AccessKeyMetadata']:
                iam_client.delete_access_key(UserName=IAM_USER_NAME, AccessKeyId=key['AccessKeyId'])
            print("  Old access keys deleted")
        else:
            raise

    ak_response = iam_client.create_access_key(UserName=IAM_USER_NAME)
    vault_access_key_id     = ak_response['AccessKey']['AccessKeyId']
    vault_secret_access_key = ak_response['AccessKey']['SecretAccessKey']
    print(f"✓ Access Key ID: {vault_access_key_id[:8]}...")

    # ─── 2. Create the role that Vault will assume through STS ──────────────
    user_arn = target_user_arn
    role_arn = f"arn:aws:iam::{account_id}:role/{ROLE_NAME}"
    # Name the bootstrap user in Principal. Same-account trust of a specific
    # IAM user ARN is a resource-based grant: STS does not require the user's
    # identity policy to allow AssumeRole, and a permissions boundary on the
    # user cannot block that grant. Account-root + aws:PrincipalArn *does*
    # require an identity-based Allow, which a boundary can deny forever.
    trust_policy = {
        "Version": "2012-10-17",
        "Statement": [{
            "Effect": "Allow",
            "Principal": {"AWS": user_arn},
            "Action": "sts:AssumeRole"
        }]
    }
    for attempt in range(1, 7):
        try:
            iam_client.create_role(
                RoleName=ROLE_NAME,
                AssumeRolePolicyDocument=json.dumps(trust_policy),
                Description='Role assumed by Vault for AWS KMS auto-unseal',
                MaxSessionDuration=3600,
                Tags=[{'Key': 'Purpose', 'Value': 'vault-auto-unseal'}]
            )
            print(f"✓ IAM role '{ROLE_NAME}' created")
            print("  Waiting for IAM role propagation...")
            time.sleep(15)
            break
        except ClientError as e:
            error_code = e.response['Error']['Code']
            if error_code == 'EntityAlreadyExists':
                iam_client.update_assume_role_policy(
                    RoleName=ROLE_NAME,
                    PolicyDocument=json.dumps(trust_policy)
                )
                print(f"✓ IAM role '{ROLE_NAME}' already exists — trust updated")
                break
            if error_code != 'MalformedPolicyDocumentException' or attempt == 6:
                raise
            print(
                f"  Waiting for IAM to accept the user as a trust principal "
                f"(attempt {attempt}/6)..."
            )
            time.sleep(10)

    # The bootstrap user can only obtain temporary credentials for this role.
    iam_client.put_user_policy(
        UserName=IAM_USER_NAME,
        PolicyName=USER_ASSUME_POLICY_NAME,
        PolicyDocument=json.dumps({
            "Version": "2012-10-17",
            "Statement": [{
                "Effect": "Allow",
                "Action": "sts:AssumeRole",
                "Resource": role_arn
            }]
        })
    )
    print("✓ Bootstrap user can assume the KMS role")

    # Remove direct KMS access left by an earlier notebook run.
    legacy_policy_arn = f"arn:aws:iam::{account_id}:policy/{POLICY_NAME}"
    try:
        iam_client.detach_user_policy(UserName=IAM_USER_NAME, PolicyArn=legacy_policy_arn)
    except ClientError as e:
        if e.response['Error']['Code'] != 'NoSuchEntity':
            raise

    # ─── 3. Create KMS Key ─────────────────────────────────────────────────
    # The key policy grants access to the assumed role, never to the user.
    kms_key_id = None
    try:
        existing = kms_client.describe_key(KeyId=KMS_ALIAS)
        metadata = existing['KeyMetadata']
        kms_key_id  = metadata['KeyId']
        kms_key_arn = metadata['Arn']
        if metadata['KeyState'] == 'PendingDeletion':
            kms_client.cancel_key_deletion(KeyId=kms_key_id)
            kms_client.enable_key(KeyId=kms_key_id)
            print('  Pending KMS deletion cancelled and key enabled')
        elif metadata['KeyState'] == 'Disabled':
            kms_client.enable_key(KeyId=kms_key_id)
            print('  KMS key enabled')
        print(f"✓ KMS key already exists — reusing: {kms_key_id}")
    except ClientError as e:
        if e.response['Error']['Code'] == 'NotFoundException':
            key_response = kms_client.create_key(
                Description='Vault Auto-Unseal Key (on-prem Vault)',
                KeyUsage='ENCRYPT_DECRYPT',
                Origin='AWS_KMS',
                Policy=json.dumps({
                    "Version": "2012-10-17",
                    "Id": "vault-auto-unseal-key-policy",
                    # Create with a stable principal, then add the role below
                    # with retry logic for IAM propagation.
                    "Statement": [{
                        "Sid": "EnableRootAccountFullAccess",
                        "Effect": "Allow",
                        "Principal": {"AWS": f"arn:aws:iam::{account_id}:root"},
                        "Action": "kms:*",
                        "Resource": "*"
                    }]
                }),
                Tags=[{'TagKey': 'Purpose', 'TagValue': 'vault-auto-unseal'}]
            )
            kms_key_id  = key_response['KeyMetadata']['KeyId']
            kms_key_arn = key_response['KeyMetadata']['Arn']
            print(f"✓ KMS Key ID : {kms_key_id}")
            print(f"  KMS Key ARN: {kms_key_arn}")

            kms_client.create_alias(AliasName=KMS_ALIAS, TargetKeyId=kms_key_id)
            print(f"✓ KMS alias '{KMS_ALIAS}' created")
        else:
            raise

    # Replace any legacy user grant with an explicit grant to the role.
    key_policy = {
        "Version": "2012-10-17",
        "Id": "vault-auto-unseal-key-policy",
        "Statement": [
            {
                "Sid": "EnableRootAccountFullAccess",
                "Effect": "Allow",
                "Principal": {"AWS": f"arn:aws:iam::{account_id}:root"},
                "Action": "kms:*",
                "Resource": "*"
            },
            {
                "Sid": "AllowVaultAutoUnsealRole",
                "Effect": "Allow",
                "Principal": {"AWS": role_arn},
                "Action": ["kms:Encrypt", "kms:Decrypt", "kms:DescribeKey"],
                "Resource": "*"
            }
        ]
    }
    for attempt in range(1, 7):
        try:
            kms_client.put_key_policy(
                KeyId=kms_key_id, PolicyName='default',
                Policy=json.dumps(key_policy)
            )
            break
        except ClientError as e:
            if e.response['Error']['Code'] != 'MalformedPolicyDocumentException' or attempt == 6:
                raise
            print(f"  Waiting for KMS to recognize the role (attempt {attempt}/6)...")
            time.sleep(10)

    # ─── 4. Attach the KMS policy to the assumed role ───────────────────────
    kms_policy_document = {
        "Version": "2012-10-17",
        "Statement": [{
            "Effect": "Allow",
            "Action": ["kms:Encrypt", "kms:Decrypt", "kms:DescribeKey"],
            "Resource": kms_key_arn
        }]
    }

    kms_policy_arn = f"arn:aws:iam::{account_id}:policy/{POLICY_NAME}"
    try:
        kms_policy_response = iam_client.create_policy(
            PolicyName=POLICY_NAME,
            Description='Allows KMS operations for Vault auto-unseal (on-prem Vault)',
            PolicyDocument=json.dumps(kms_policy_document)
        )
        kms_policy_arn = kms_policy_response['Policy']['Arn']
        print(f"✓ KMS policy created: {kms_policy_arn}")
    except ClientError as e:
        if e.response['Error']['Code'] == 'EntityAlreadyExists':
            # Reusing an ARN is not enough: update the document so it
            # always points to the current KMS key.
            versions = iam_client.list_policy_versions(PolicyArn=kms_policy_arn)['Versions']
            for version in versions:
                if not version['IsDefaultVersion']:
                    iam_client.delete_policy_version(
                        PolicyArn=kms_policy_arn, VersionId=version['VersionId']
                    )
            iam_client.create_policy_version(
                PolicyArn=kms_policy_arn,
                PolicyDocument=json.dumps(kms_policy_document),
                SetAsDefault=True
            )
            print(f"✓ KMS policy '{POLICY_NAME}' updated and reused")
        else:
            raise

    try:
        iam_client.attach_role_policy(RoleName=ROLE_NAME, PolicyArn=kms_policy_arn)
        print(f"✓ KMS policy attached to role")
    except ClientError:
        pass  # already attached

    # ─── 5. Validate AssumeRole and persist values ──────────────────────────
    bootstrap_sts = boto3.client(
        'sts',
        aws_access_key_id=vault_access_key_id,
        aws_secret_access_key=vault_secret_access_key
    )
    retryable_sts_errors = {'AccessDenied', 'InvalidClientTokenId'}
    for attempt in range(1, 13):
        try:
            assumed = bootstrap_sts.assume_role(
                RoleArn=role_arn,
                RoleSessionName='vault-auto-unseal-validation'
            )['Credentials']
            break
        except ClientError as e:
            error_code = e.response['Error']['Code']
            if error_code not in retryable_sts_errors or attempt == 12:
                raise
            print(
                f"  STS returned {error_code}; waiting for access-key/policy "
                f"propagation (attempt {attempt}/12)..."
            )
            time.sleep(10)
    boto3.client(
        'kms', region_name=REGION,
        aws_access_key_id=assumed['AccessKeyId'],
        aws_secret_access_key=assumed['SecretAccessKey'],
        aws_session_token=assumed['SessionToken']
    ).describe_key(KeyId=kms_key_id)
    os.environ['KMS_KEY_ID']                  = kms_key_id
    os.environ['REGION']                      = REGION
    os.environ['VAULT_AWS_ACCESS_KEY_ID']     = vault_access_key_id
    os.environ['VAULT_AWS_SECRET_ACCESS_KEY'] = vault_secret_access_key
    os.environ['VAULT_AWS_ROLE_ARN']           = role_arn

    print(f"\n✓ All values stored in environment")
    print(f"  KMS Key ARN: {kms_key_arn}")
    print(f"  Role ARN: {role_arn}")
    print("  ✓ AssumeRole + KMS access validated")
    print("  Bootstrap credentials will be mounted as a shared credentials file")

except ClientError as e:
    error_code = e.response['Error']['Code']
    error_msg  = e.response['Error']['Message']
    print(f"\n✗ AWS API error [{error_code}]: {error_msg}")
    print("  Resources are idempotently reused; cleanup is not required before retrying.")
    raise
except Exception as e:
    print(f"\n✗ Unexpected error: {e}")
    raise

## 3. Create K8S cluster and preload Vault Enterprise

The Vault Enterprise image is pulled by Podman on macOS, exported as a Docker archive, and loaded into the Minikube node. This avoids relying on Docker Hub DNS/connectivity from inside the Minikube VM. Helm uses `IfNotPresent`, so the preloaded image is selected without another registry pull.

In [ ]:
! minikube delete -p workshop2

In [ ]:
! open -a Podman\ Desktop

In [ ]:
! minikube start -p workshop2 --driver=podman

In [ ]:
%%bash
helm repo add hashicorp https://helm.releases.hashicorp.com
helm repo update

In [ ]:
%env WORKDIR=/tmp/vault
%env VAULT_K8S_NAMESPACE=vault
%env VAULT_HELM_RELEASE_NAME=vault
%env VAULT_SERVICE_NAME=vault-internal 
%env K8S_CLUSTER_NAME=cluster.local 

In [ ]:
%%bash
rm -rf /tmp/vault
mkdir /tmp/vault

In [ ]:
! kubectl create namespace $VAULT_K8S_NAMESPACE

## 4. Create the AWS shared credentials Secret

Store an AWS shared credentials file in a Kubernetes Secret. `[default]` holds the bootstrap user's long-lived key for the inner AssumeRole session. `[vault-bootstrap]` has no `aws_access_key_id`, so the wrapper's shared-credentials provider cannot win and skip STS.

In [ ]:
%%bash
# Create an AWS shared credentials file without exposing credentials in args.
mkdir -p "${WORKDIR}/aws"
umask 077
cat > "${WORKDIR}/aws/credentials" <<EOF
[default]
aws_access_key_id=${VAULT_AWS_ACCESS_KEY_ID}
aws_secret_access_key=${VAULT_AWS_SECRET_ACCESS_KEY}

[vault-bootstrap]
role_arn=${VAULT_AWS_ROLE_ARN}
role_session_name=vault-auto-unseal
source_profile=default
EOF

kubectl delete secret vault-aws-creds --namespace "${VAULT_K8S_NAMESPACE}" --ignore-not-found
kubectl create secret generic vault-aws-creds \
  --namespace "${VAULT_K8S_NAMESPACE}" \
  --from-file=credentials="${WORKDIR}/aws/credentials"

echo "✓ Secret 'vault-aws-creds' created in namespace '${VAULT_K8S_NAMESPACE}'"

## 5. Generate TLS certificates

In [ ]:
%%bash

openssl genrsa -out ${WORKDIR}/vault.key 2048
cat > ${WORKDIR}/vault-csr.conf <<EOF
[req]
default_bits = 2048
prompt = no
encrypt_key = yes
default_md = sha256
distinguished_name = kubelet_serving
req_extensions = v3_req
[ kubelet_serving ]
O = system:nodes
CN = system:node:*.${VAULT_HELM_RELEASE_NAME}.svc.${K8S_CLUSTER_NAME}
[ v3_req ]
basicConstraints = CA:FALSE
keyUsage = nonRepudiation, digitalSignature, keyEncipherment, dataEncipherment
extendedKeyUsage = serverAuth, clientAuth
subjectAltName = @alt_names
[alt_names]
DNS.1 = *.${VAULT_SERVICE_NAME}
DNS.2 = *.${VAULT_SERVICE_NAME}.${VAULT_HELM_RELEASE_NAME}.svc.${K8S_CLUSTER_NAME}
DNS.3 = *.${VAULT_HELM_RELEASE_NAME}
DNS.4 = *.${VAULT_HELM_RELEASE_NAME}.svc.${K8S_CLUSTER_NAME}
IP.1 = 127.0.0.1
EOF

openssl req -new -key ${WORKDIR}/vault.key -out ${WORKDIR}/vault.csr -config ${WORKDIR}/vault-csr.conf


cat > ${WORKDIR}/csr.yaml <<EOF
apiVersion: certificates.k8s.io/v1
kind: CertificateSigningRequest
metadata:
   name: vault.svc
spec:
   signerName: kubernetes.io/kubelet-serving
   expirationSeconds: 8640000
   request: $(cat ${WORKDIR}/vault.csr|base64|tr -d '\n')
   usages:
   - digital signature
   - key encipherment
   - server auth
EOF

kubectl create -f ${WORKDIR}/csr.yaml
kubectl certificate approve vault.svc
kubectl get csr vault.svc
kubectl get csr vault.svc -o jsonpath='{.status.certificate}' | openssl base64 -d -A -out ${WORKDIR}/vault.crt
kubectl config view \
--raw \
--minify \
--flatten \
-o jsonpath='{.clusters[].cluster.certificate-authority-data}' \
| base64 -d > ${WORKDIR}/vault.ca



kubectl create secret generic vault-ha-tls \
   -n $VAULT_K8S_NAMESPACE \
   --from-file=vault.key=${WORKDIR}/vault.key \
   --from-file=vault.crt=${WORKDIR}/vault.crt \
   --from-file=vault.ca=${WORKDIR}/vault.ca

In [ ]:
%%bash
secret=$(cat vault.hclic)
kubectl create secret generic vault-ent-license --from-literal="license=${secret}" -n $VAULT_K8S_NAMESPACE

## 6. Helm overrides — On-Prem Vault with AWS KMS auto-unseal

Key points:
- The Secret is mounted at `/vault/userconfig/aws/credentials`
- `shared_creds_profile` is `vault-bootstrap`, which has no static keys
- `role_arn` in the stanza enables AssumeRole; `AWS_SHARED_CREDENTIALS_FILE` lets that session read `[default]`
- The user can only assume the role; only the role can use KMS

In [ ]:
%%bash
cat > ${WORKDIR}/overrides.yaml <<EOF
global:
   enabled: true
   tlsDisable: false

csi:
   enabled: false

injector:
   enabled: false

logLevel: "trace"

server:
   image:
      repository: docker.io/hashicorp/vault-enterprise
      tag: 1.19.16-ent
      pullPolicy: IfNotPresent
   enterpriseLicense:
      secretName: vault-ent-license

   extraEnvironmentVars:
      VAULT_CACERT: /vault/userconfig/vault-ha-tls/vault.ca
      VAULT_TLSCERT: /vault/userconfig/vault-ha-tls/vault.crt
      VAULT_TLSKEY: /vault/userconfig/vault-ha-tls/vault.key
      AWS_SHARED_CREDENTIALS_FILE: /vault/userconfig/aws/credentials

   volumes:
      - name: userconfig-vault-ha-tls
        secret:
         defaultMode: 420
         secretName: vault-ha-tls
      - name: aws-shared-credentials
        secret:
         defaultMode: 256
         secretName: vault-aws-creds

   volumeMounts:
      - mountPath: /vault/userconfig/vault-ha-tls
        name: userconfig-vault-ha-tls
        readOnly: true
      - mountPath: /vault/userconfig/aws
        name: aws-shared-credentials
        readOnly: true

   standalone:
      enabled: false
   affinity: ""
   ha:
      enabled: true
      replicas: 3
      raft:
         enabled: true
         setNodeId: true
         config: |
            ui = true
            listener "tcp" {
               tls_disable = 0
               address = "[::]:8200"
               cluster_address = "[::]:8201"
               tls_cert_file = "/vault/userconfig/vault-ha-tls/vault.crt"
               tls_key_file  = "/vault/userconfig/vault-ha-tls/vault.key"
               tls_client_ca_file = "/vault/userconfig/vault-ha-tls/vault.ca"
            }
            storage "raft" {
               path = "/vault/data"

               retry_join {
                  auto_join             = "provider=k8s namespace=vault label_selector=\"component=server,app.kubernetes.io/name=vault\""
                  auto_join_scheme      = "https"
                  leader_ca_cert_file   = "/vault/userconfig/vault-ha-tls/vault.ca"
                  leader_tls_servername = "vault-0.vault-internal"
               }

            }
            # vault-bootstrap has no static keys, so SharedCredentialsProvider
            # fails. role_arn then adds AssumeRole, which reads [default] via
            # AWS_SHARED_CREDENTIALS_FILE. Do not set AWS_PROFILE.
            seal "awskms" {
               region                = "${REGION}"
               kms_key_id            = "${KMS_KEY_ID}"
               shared_creds_filename = "/vault/userconfig/aws/credentials"
               shared_creds_profile  = "vault-bootstrap"
               role_arn              = "${VAULT_AWS_ROLE_ARN}"
               role_session_name     = "vault-auto-unseal"
            }
            telemetry {
               disable_hostname = true
               prometheus_retention_time = "12h"
            }
            disable_mlock = true
            service_registration "kubernetes" {}

   ui:
      enabled: true
      serviceType: "LoadBalancer"
      serviceNodePort: null
      externalPort: 8200

EOF

echo "✓ overrides.yaml written to ${WORKDIR}/overrides.yaml"
cat ${WORKDIR}/overrides.yaml

## 7. Deploy Vault with Helm

In [ ]:
%%bash
helm upgrade --install ${VAULT_HELM_RELEASE_NAME} hashicorp/vault \
  --namespace ${VAULT_K8S_NAMESPACE} \
  --values ${WORKDIR}/overrides.yaml

echo ""
echo "✓ Helm release '${VAULT_HELM_RELEASE_NAME}' deployed"
echo "  Waiting for pods to be scheduled..."
kubectl get pods -n ${VAULT_K8S_NAMESPACE}

### Verify the shared credentials mount and STS/KMS initialization
Confirm the profile file is mounted without printing its contents, then inspect Vault startup logs for authentication or KMS errors.

In [ ]:
%%bash
# Phase=Running is true even during CrashLoopBackOff; the vault process may
# not be up yet. Logs are the source of truth for seal/KMS errors.
kubectl wait pod vault-0 \
  --namespace ${VAULT_K8S_NAMESPACE} \
  --for=jsonpath='{.status.phase}'=Running \
  --timeout=120s || true

echo "=== Pod status ==="
kubectl get pod vault-0 -n ${VAULT_K8S_NAMESPACE} -o wide

echo ""
echo "=== AWS shared credentials mount ==="
kubectl exec -n ${VAULT_K8S_NAMESPACE} vault-0 -- \
  sh -c 'test -r /vault/userconfig/aws/credentials && echo "credentials profile mounted and readable"' \
  || echo "(container not running yet; check logs below)"

echo ""
echo "=== Vault logs (seal / KMS) ==="
kubectl logs vault-0 -n ${VAULT_K8S_NAMESPACE} --tail=80 2>/dev/null | \
  grep -E -i 'seal|awskms|assume|sts|kms|error|credential' || \
  kubectl logs vault-0 -n ${VAULT_K8S_NAMESPACE} --previous --tail=80 2>/dev/null | \
    grep -E -i 'seal|awskms|assume|sts|kms|error|credential' || true

## 8. Initialize Vault

With AWS KMS auto-unseal, Vault only needs to be **initialized** once. The `[default]` keys cannot use KMS; only the assumed role can. Recovery keys replace Shamir unseal keys for recovery operations.

In [ ]:
%%bash
# Give Vault a few seconds to finish starting its listener
sleep 5

echo "--- Initializing Vault (recovery-shares=1, recovery-threshold=1) ---"
kubectl exec -n ${VAULT_K8S_NAMESPACE} vault-0 -- \
  vault operator init \
  -recovery-shares=1 \
  -recovery-threshold=1 \
  -format=json > ${WORKDIR}/vault-init.json

echo ""
echo "✓ Init output saved securely to ${WORKDIR}/vault-init.json (not printed)"

In [ ]:
import json, os, re

try:
    with open('/tmp/vault/vault-init.json') as f:
        raw = f.read()

    if not raw.strip():
        raise ValueError("vault-init.json is empty – Vault init may have failed. Check the init cell output.")

    match = re.search(r'\{', raw)
    if match:
        raw = raw[match.start():]

    init_data = json.loads(raw)

    root_token   = init_data['root_token']
    recovery_key = init_data['recovery_keys_b64'][0]

    os.environ['VAULT_TOKEN'] = root_token
    print('✓ Root token loaded into VAULT_TOKEN without displaying it')
    print('✓ Recovery key present in the protected init file; store it securely')

except FileNotFoundError:
    print("✗ /tmp/vault/vault-init.json not found – run the Vault init cell first.")
except (json.JSONDecodeError, ValueError) as e:
    print(f"✗ Could not parse vault-init.json: {e}")
    print("  Re-run the Vault init cell and check its output for errors.")
except (KeyError, IndexError) as e:
    print(f"✗ Unexpected init JSON structure: {e}")
    raise

In [ ]:
%%bash
echo "=== Wait for the three-node Raft cluster ==="
kubectl wait pod -n ${VAULT_K8S_NAMESPACE} -l app.kubernetes.io/name=vault \
  --for=condition=Ready --timeout=240s
kubectl get pods -n ${VAULT_K8S_NAMESPACE} -l app.kubernetes.io/name=vault

echo "=== Restart vault-0 and prove credential reacquisition + auto-unseal ==="
kubectl delete pod -n ${VAULT_K8S_NAMESPACE} vault-0
kubectl wait pod/vault-0 -n ${VAULT_K8S_NAMESPACE} \
  --for=condition=Ready --timeout=240s
kubectl exec -n ${VAULT_K8S_NAMESPACE} vault-0 -- \
  vault status -tls-skip-verify
echo "✓ vault-0 restarted and became Ready without manual unseal"

## 9. Verify unseal used STS AssumeRole (CloudTrail)

KMS `Encrypt` / `Decrypt` / `DescribeKey` in `eu-west-3` are the proof. If `userIdentity.type` is `AssumedRole` and the ARN contains `vault-kms-unseal`, Vault unsealed with the role. If the type is `IAMUser` / `vault-autounseal`, KMS ran as the bootstrap user.

To confirm a **new** STS token (not reuse of the same session), compare `userIdentity.accessKeyId` (`ASIA…`) across events. Restart `vault-0` and re-run this cell: a second successful `AssumeRole` plus KMS calls with a **different** `ASIA` key means a new token. Vault 2.1.0 snapshots the STS credentials at process start (`awsutil` v0.3.0 `StaticProvider`), so a live pod will not call `AssumeRole` again until it restarts; the role session lasts 1 hour.

In [ ]:
import json
import os
from datetime import datetime, timedelta, timezone

import boto3
from botocore.exceptions import ClientError

REGION = os.environ.get('REGION', 'eu-west-3')
ROLE_NAME = 'vault-kms-unseal'
USER_NAME = 'vault-autounseal'
SESSION_NAME = 'vault-auto-unseal'
KMS_NAMES = {'DescribeKey', 'Encrypt', 'Decrypt', 'GenerateDataKey'}
STS_NAMES = {'AssumeRole'}

end = datetime.now(timezone.utc)
start = end - timedelta(hours=4)


def _summarize(event):
    uid = event.get('userIdentity') or {}
    issuer = (uid.get('sessionContext') or {}).get('sessionIssuer') or {}
    params = event.get('requestParameters') or {}
    return {
        'time': event.get('eventTime'),
        'region': event.get('awsRegion'),
        'event': event.get('eventName'),
        'principal_type': uid.get('type'),
        'principal_arn': uid.get('arn'),
        'user_name': uid.get('userName'),
        'session_issuer': issuer.get('userName') or issuer.get('arn'),
        'assumed_role': params.get('roleArn'),
        'role_session_name': params.get('roleSessionName'),
        'access_key_id': uid.get('accessKeyId'),
        'error': event.get('errorCode'),
        'event_id': event.get('eventID'),
    }


def _lookup_attr(region, key, value):
    client = boto3.client('cloudtrail', region_name=region)
    rows = []
    try:
        paginator = client.get_paginator('lookup_events')
        for page in paginator.paginate(
            LookupAttributes=[{'AttributeKey': key, 'AttributeValue': value}],
            StartTime=start,
            EndTime=end,
        ):
            for item in page.get('Events', []):
                rows.append(json.loads(item['CloudTrailEvent']))
    except ClientError as exc:
        print(
            f"✗ CloudTrail LookupEvents failed in {region} "
            f"({key}={value}): {exc.response['Error']['Code']}: "
            f"{exc.response['Error']['Message']}"
        )
    return rows


def _collect(region, names):
    seen = set()
    rows = []
    queries = [
        ('Username', USER_NAME),
        ('Username', ROLE_NAME),
        ('Username', SESSION_NAME),
    ]
    kms_key_id = os.environ.get('KMS_KEY_ID')
    role_arn = os.environ.get('VAULT_AWS_ROLE_ARN')
    if kms_key_id:
        queries.append(('ResourceName', kms_key_id))
    if role_arn:
        queries.append(('ResourceName', role_arn))

    for key, value in queries:
        for raw in _lookup_attr(region, key, value):
            event_id = raw.get('eventID')
            if event_id in seen or raw.get('eventName') not in names:
                continue
            seen.add(event_id)
            rows.append(_summarize(raw))
    rows.sort(key=lambda row: row.get('time') or '')
    return rows


print(f"Looking up CloudTrail from {start.isoformat()} to {end.isoformat()}")
print(f"KMS key: {os.environ.get('KMS_KEY_ID', '(KMS_KEY_ID not set — run the provisioning cell)')}")
print(f"Role:    {os.environ.get('VAULT_AWS_ROLE_ARN', '(VAULT_AWS_ROLE_ARN not set)')}")
print()

kms_events = _collect(REGION, KMS_NAMES)
sts_events = _collect(REGION, STS_NAMES) + _collect('us-east-1', STS_NAMES)

print('=== STS AssumeRole ===')
if not sts_events:
    print('  (no matching events yet — CloudTrail can lag 5–15 minutes)')
for event in sts_events:
    print(
        f"  {event['time']}  {event['region']}  {event['principal_type']}  "
        f"{event['principal_arn']}"
    )
    print(
        f"    assumed {event['assumed_role']}  "
        f"session={event['role_session_name']}  "
        f"sts_key={event['access_key_id']}  error={event['error']}"
    )

print('\n=== KMS cryptographic API calls ===')
if not kms_events:
    print('  (no matching events yet — CloudTrail can lag 5–15 minutes)')
for event in kms_events:
    print(f"  {event['time']}  {event['event']}  {event['principal_type']}")
    print(f"    arn={event['principal_arn']}")
    print(
        f"    issuer={event['session_issuer']}  "
        f"sts_key={event['access_key_id']}  error={event['error']}"
    )

assumed_kms = [
    event for event in kms_events
    if event['principal_type'] == 'AssumedRole'
    and ROLE_NAME in (event['principal_arn'] or '')
]
user_kms = [
    event for event in kms_events
    if event['principal_type'] == 'IAMUser'
    and USER_NAME in ((event['user_name'] or '') + (event['principal_arn'] or ''))
]
sts_ok = [
    event for event in sts_events
    if ROLE_NAME in (event.get('assumed_role') or '') and not event.get('error')
]

print('\n=== Distinct STS access keys used for KMS (AssumedRole vault-kms-unseal) ===')
token_times = {}
for event in assumed_kms:
    key = event.get('access_key_id')
    if not key:
        continue
    token_times.setdefault(key, []).append(event['time'])
if not token_times:
    print('  (none)')
for key, times in sorted(token_times.items(), key=lambda item: min(item[1])):
    print(f"  {key}  first={min(times)}  last={max(times)}  kms_calls={len(times)}")
print(
    "  A new ASIA… key after a pod restart means Vault obtained a new STS token. "
    "The same ASIA… key across an hour without restart means the process reused "
    "the snapshot from awsutil v0.3.0 (no in-process refresh)."
)
if assumed_kms:
    print(
        f"✓ KMS was called as AssumedRole {ROLE_NAME} "
        f"({len(assumed_kms)} event(s)). Unseal used STS."
    )
elif user_kms:
    print(
        f"✗ KMS was called as IAM user {USER_NAME}. "
        "AssumeRole did not take effect for KMS."
    )
elif sts_ok:
    print('~ AssumeRole succeeded, but no KMS events yet. Wait a few minutes and re-run.')
else:
    print(
        '? No matching CloudTrail events yet. '
        'Wait 5–15 minutes after init/restart and re-run this cell.'
    )

# Clean up

In [ ]:
%%bash
# 1 – Complete local cleanup before touching AWS
helm uninstall ${VAULT_HELM_RELEASE_NAME} --namespace ${VAULT_K8S_NAMESPACE} || true
kubectl delete namespace ${VAULT_K8S_NAMESPACE} --ignore-not-found --wait=true --timeout=180s || true
minikube delete -p workshop2 || true
rm -f /tmp/vault-enterprise-2.1.0-ent-assumerole.tar
rm -rf "${WORKDIR}"
echo "✓ Vault, namespace, Minikube profile, Podman image, and temporary secrets removed"

In [ ]:
# 2 – Remove all AWS resources created by this notebook
import boto3, os
from botocore.exceptions import ClientError

REGION        = os.environ.get('REGION', 'eu-west-3')
IAM_USER_NAME = 'vault-autounseal'
POLICY_NAME   = 'vault-autounseal-kms-policy'
KMS_ALIAS     = 'alias/vault-auto-unseal'

iam = boto3.client('iam')
kms = boto3.client('kms', region_name=REGION)
sts = boto3.client('sts')
account_id = sts.get_caller_identity()['Account']
kms_policy_arn = f"arn:aws:iam::{account_id}:policy/{POLICY_NAME}"

# Delete access keys for the user
try:
    for key in iam.list_access_keys(UserName=IAM_USER_NAME)['AccessKeyMetadata']:
        iam.delete_access_key(UserName=IAM_USER_NAME, AccessKeyId=key['AccessKeyId'])
    print("✓ Access keys deleted")
except ClientError as e:
    print(f"⚠ Access keys: {e.response['Error']['Message']}")

# Detach ALL attached policies from user (covers renamed/moved policies)
try:
    attached = iam.list_attached_user_policies(UserName=IAM_USER_NAME)['AttachedPolicies']
    for pol in attached:
        iam.detach_user_policy(UserName=IAM_USER_NAME, PolicyArn=pol['PolicyArn'])
        print(f"✓ Detached policy '{pol['PolicyName']}' from user")
    if not attached:
        print("  No attached policies found on user")
except ClientError as e:
    print(f"⚠ Detach policies: {e.response['Error']['Message']}")

# Delete user
try:
    for policy_name in iam.list_user_policies(UserName=IAM_USER_NAME)['PolicyNames']:
        iam.delete_user_policy(UserName=IAM_USER_NAME, PolicyName=policy_name)
        print(f"✓ Deleted inline user policy '{policy_name}'")
    iam.delete_user(UserName=IAM_USER_NAME)
    print(f"✓ IAM user '{IAM_USER_NAME}' deleted")
except ClientError as e:
    print(f"⚠ IAM user: {e.response['Error']['Message']}")

# Detach policy from ALL remaining entities, remove old versions, then delete
try:
    entities = iam.list_entities_for_policy(PolicyArn=kms_policy_arn)
    for u in entities.get('PolicyUsers', []):
        iam.detach_user_policy(UserName=u['UserName'], PolicyArn=kms_policy_arn)
        print(f"  Detached policy from user '{u['UserName']}'")
    for g in entities.get('PolicyGroups', []):
        iam.detach_group_policy(GroupName=g['GroupName'], PolicyArn=kms_policy_arn)
        print(f"  Detached policy from group '{g['GroupName']}'")
    for r in entities.get('PolicyRoles', []):
        iam.detach_role_policy(RoleName=r['RoleName'], PolicyArn=kms_policy_arn)
        print(f"  Detached policy from role '{r['RoleName']}'")
    for version in iam.list_policy_versions(PolicyArn=kms_policy_arn)['Versions']:
        if not version['IsDefaultVersion']:
            iam.delete_policy_version(
                PolicyArn=kms_policy_arn, VersionId=version['VersionId']
            )
    iam.delete_policy(PolicyArn=kms_policy_arn)
    print(f"✓ IAM policy '{POLICY_NAME}' deleted")
except ClientError as e:
    print(f"⚠ Policy: {e.response['Error']['Message']}")

# Delete the AssumeRole role after all managed and inline policies are removed
ROLE_NAME = 'vault-kms-unseal'
try:
    for policy_name in iam.list_role_policies(RoleName=ROLE_NAME)['PolicyNames']:
        iam.delete_role_policy(RoleName=ROLE_NAME, PolicyName=policy_name)
    for policy in iam.list_attached_role_policies(RoleName=ROLE_NAME)['AttachedPolicies']:
        iam.detach_role_policy(RoleName=ROLE_NAME, PolicyArn=policy['PolicyArn'])
    iam.delete_role(RoleName=ROLE_NAME)
    print(f"✓ IAM role '{ROLE_NAME}' deleted")
except ClientError as e:
    print(f"⚠ IAM role: {e.response['Error']['Message']}")

# Schedule KMS key deletion
try:
    alias_info = kms.describe_key(KeyId=KMS_ALIAS)
    kms_key_id = alias_info['KeyMetadata']['KeyId']
    kms.delete_alias(AliasName=KMS_ALIAS)
    kms.schedule_key_deletion(KeyId=kms_key_id, PendingWindowInDays=7)
    print(f"✓ KMS key '{kms_key_id}' scheduled for deletion in 7 days")
except ClientError as e:
    print(f"⚠ KMS key: {e.response['Error']['Message']}")

print("\n✓ AWS cleanup complete")

In [ ]:
%%bash
# Local cleanup intentionally runs before the AWS cleanup cell.
echo "✓ Cleanup order: local environment first, AWS resources second"